# Merged Dataset Inspection
## APPA-EEA Data Quality Analysis

This notebook provides a comprehensive inspection of the merged APPA-EEA dataset by proximity, with a focus on identifying and understanding missing data (NaN values).

## 1. Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set style for better visualizations
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 8)

## 2. Load the Merged Dataset

In [ ]:
# Load the merged dataset
df = pd.read_csv('../output/merged_appa_eea_by_proximity.csv')

print(f"Dataset loaded successfully!")
print(f"File: merged_appa_eea_by_proximity.csv")

In [ ]:
pd.set_option('display.max_columns', None)
df.head(5000)

## 3. Dataset Overview

In [ ]:
# Display basic dataset information
print("="*80)
print("DATASET SHAPE AND SIZE")
print("="*80)
print(f"Total rows: {df.shape[0]:,}")
print(f"Total columns: {df.shape[1]}")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

print("\n" + "="*80)
print("FIRST 5 ROWS")
print("="*80)
print(df.head())

print("\n" + "="*80)
print("LAST 5 ROWS")
print("="*80)
print(df.tail())

print("\n" + "="*80)
print("BASIC STATISTICS")
print("="*80)
print(df.describe())

## 4. Data Types and Structure

In [ ]:
# Display data types and structure
print("="*80)
print("DATA TYPES")
print("="*80)
print(df.dtypes)

print("\n" + "="*80)
print("DETAILED DATASET INFO")
print("="*80)
df.info()

## 5. Missing Values Analysis (NaN) 🔍

In [ ]:
# Comprehensive NaN analysis
print("="*80)
print("MISSING VALUES SUMMARY")
print("="*80)

# Count and percentage of NaN values
missing_data = pd.DataFrame({
    'Column': df.columns,
    'Missing_Count': df.isnull().sum(),
    'Missing_Percent': (df.isnull().sum() / len(df) * 100).round(2),
    'Data_Type': df.dtypes
})

# Sort by missing percentage (descending)
missing_data = missing_data.sort_values('Missing_Percent', ascending=False)

print(missing_data.to_string(index=False))

print("\n" + "="*80)
print("OVERALL STATISTICS")
print("="*80)
total_cells = df.shape[0] * df.shape[1]
total_missing = df.isnull().sum().sum()
total_missing_pct = (total_missing / total_cells * 100)

print(f"Total cells in dataset: {total_cells:,}")
print(f"Total missing cells: {total_missing:,}")
print(f"Overall missing percentage: {total_missing_pct:.2f}%")
print(f"\nColumns with NO missing values: {(missing_data['Missing_Count'] == 0).sum()}")
print(f"Columns with SOME missing values: {(missing_data['Missing_Count'] > 0).sum()}")
print(f"Columns with ALL missing values: {(missing_data['Missing_Percent'] == 100).sum()}")

## 7. Detect Problematic Columns

In [20]:
# Identify problematic columns
print("="*80)
print("PROBLEMATIC COLUMNS ANALYSIS")
print("="*80)

# Columns with >50% missing
high_missing = missing_data[missing_data['Missing_Percent'] > 50]
print(f"\n  Columns with >50% missing data ({len(high_missing)}):")
if len(high_missing) > 0:
    print(high_missing[['Column', 'Missing_Percent']].to_string(index=False))
else:
    print("  None")

# Columns with 100% missing (completely empty)
all_missing = missing_data[missing_data['Missing_Percent'] == 100]
print(f"\n Columns with 100% missing data ({len(all_missing)}):")
if len(all_missing) > 0:
    print(all_missing[['Column', 'Missing_Percent']].to_string(index=False))
    print("\n  Recommendation: Consider dropping these columns")
else:
    print("  None")

# Columns with some missing (0-50%)
some_missing = missing_data[(missing_data['Missing_Percent'] > 0) & (missing_data['Missing_Percent'] <= 50)]
print(f"\n⚡ Columns with some missing data (0-50%) ({len(some_missing)}):")
if len(some_missing) > 0:
    print(some_missing[['Column', 'Missing_Percent']].to_string(index=False))
else:
    print("  None")



PROBLEMATIC COLUMNS ANALYSIS

  Columns with >50% missing data (155):
                                              Column  Missing_Percent
       SPO.IT0842A_5_BETA_2022-12-01_00:00:00_Valore            83.57
   SPO.IT0842A_5_BETA_2022-12-01_00:00:00_Latitudine            83.57
  SPO.IT0842A_5_BETA_2022-12-01_00:00:00_Longitudine            83.57
   SPO.IT0842A_5_BETA_2022-12-01_00:00:00_Inquinante            83.57
 SPO.IT0842A_5_BETA_2022-12-01_00:00:00_Unità_misura            83.57
   SPO.IT2230A_5_BETA_2023-01-20_00:00:00_Inquinante            83.57
 SPO.IT2230A_5_BETA_2023-01-20_00:00:00_Unità_misura            83.57
   SPO.IT2230A_5_BETA_2023-01-20_00:00:00_Latitudine            83.57
  SPO.IT2230A_5_BETA_2023-01-20_00:00:00_Longitudine            83.57
       SPO.IT2230A_5_BETA_2023-01-20_00:00:00_Valore            83.57
       SPO.IT0499A_5_BETA_2004-10-09_00:00:00_Valore            83.17
   SPO.IT0499A_5_BETA_2004-10-09_00:00:00_Inquinante            83.17
   SPO.IT0499A_5_BET

## 8. Summary and Recommendations

In [ ]:
# Final summary and recommendations
print("="*80)
print("FINAL SUMMARY & RECOMMENDATIONS")
print("="*80)

print(f"""
📊 DATASET QUALITY REPORT
─────────────────────────────────────
• Dataset Size: {df.shape[0]:,} rows × {df.shape[1]} columns
• Total Data Points: {df.shape[0] * df.shape[1]:,}
• Missing Data Points: {total_missing:,}
• Overall Completeness: {100 - total_missing_pct:.2f}%

🎯 ACTION ITEMS
─────────────────────────────────────
""")

if len(all_missing) > 0:
    print(f"1. DROP completely empty columns ({len(all_missing)})")
    print(f"   df = df.drop(columns={all_missing['Column'].tolist()})")

if len(high_missing) > 0:
    print(f"\n2. REVIEW high-missing columns ({len(high_missing)})")
    for col in high_missing['Column']:
        pct = high_missing[high_missing['Column'] == col]['Missing_Percent'].values[0]
        print(f"   - {col}: {pct:.1f}% missing")

if len(some_missing) > 0:
    print(f"\n3. HANDLE moderate-missing columns ({len(some_missing)}) - choose strategy:")
    print(f"   Options: drop rows, forward fill, interpolation, or impute with mean/median")

print(f"\n✅ NEXT STEPS")
print(f"─────────────────────────────────────")
print(f"1. Review the missing data patterns above")
print(f"2. Decide on handling strategy for each problematic column")
print(f"3. Consider domain knowledge when imputing or dropping data")
print(f"4. Document all data cleaning decisions for reproducibility")
